## Linear Classifier in TensorFlow 
Using Low Level API in Eager Execution mode

### Load tensorflow

In [85]:
import tensorflow as tf

In [86]:
tf.__version__

'1.14.0'

In [87]:
#Enable Eager Execution if using tensflow version < 2.0
#From tensorflow v2.0 onwards, Eager Execution will be enabled by default
tf.enable_eager_execution()

#Check eager execution is enabled or not
tf.executing_eagerly()

True

### Collect Data

In [88]:
# Im running in my local pc
#from google.colab import drive
#drive.mount('/gdrive')

In [89]:
import pandas as pd

In [90]:
data = pd.read_csv('prices.csv')

### Check all columns in the dataset

In [91]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 851264 entries, 0 to 851263
Data columns (total 7 columns):
date      851264 non-null object
symbol    851264 non-null object
open      851264 non-null float64
close     851264 non-null float64
low       851264 non-null float64
high      851264 non-null float64
volume    851264 non-null float64
dtypes: float64(5), object(2)
memory usage: 45.5+ MB


### Drop columns `date` and  `symbol`

In [92]:
data.drop(['date', 'symbol'], axis=1, inplace=True)

In [93]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 851264 entries, 0 to 851263
Data columns (total 5 columns):
open      851264 non-null float64
close     851264 non-null float64
low       851264 non-null float64
high      851264 non-null float64
volume    851264 non-null float64
dtypes: float64(5)
memory usage: 32.5 MB


In [94]:
data.head()

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2163600.0
1,125.239998,119.980003,119.940002,125.540001,2386400.0
2,116.379997,114.949997,114.930000,119.739998,2489500.0
3,115.480003,116.620003,113.500000,117.440002,2006300.0
4,117.010002,114.970001,114.089996,117.330002,1408600.0


### Consider only first 1000 rows in the dataset for building feature set and target set
Target 'Volume' has very high values. Divide 'Volume' by 1000,000

In [95]:
data.shape

(851264, 5)

In [96]:
data = data[0:1000]
data.shape

(1000, 5)

In [97]:
data['volume'] = data['volume']/1000000

In [98]:
data.head()

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2.1636
1,125.239998,119.980003,119.940002,125.540001,2.3864
2,116.379997,114.949997,114.930000,119.739998,2.4895
3,115.480003,116.620003,113.500000,117.440002,2.0063
4,117.010002,114.970001,114.089996,117.330002,1.4086


### Divide the data into train and test sets

In [99]:
from sklearn.model_selection import train_test_split

In [100]:
X =  data.drop("volume", axis=1)
Y =  data.pop("volume")

In [101]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.33, random_state=7)

#### Convert Training and Test Data to numpy float32 arrays


In [102]:
import numpy as np

In [103]:
xtr = np.asarray(X_train, dtype='float32')
xte = np.asarray(X_test, dtype='float32')
ytr = np.asarray(y_train, dtype='float32')
yte = np.asarray(y_test, dtype='float32')

In [104]:
xtr.shape, X_train.shape

((670, 4), (670, 4))

### Normalize the data
You can use Normalizer from sklearn.preprocessing

In [105]:
from sklearn.preprocessing import Normalizer

In [106]:
xtr = Normalizer().fit_transform(xtr)
xte = Normalizer().fit_transform(xte)

## Building the Model in tensorflow

1.Define Weights and Bias, use tf.zeros to initialize weights and Bias

In [124]:
W = tf.Variable(tf.random_normal(shape=[4, 1]), name="Weights")
b = tf.Variable(tf.random_normal(shape=[1]), name="Bias")

2.Define a function to calculate prediction

In [125]:
def pred(x, w, b):
  return tf.add(tf.matmul(x, W),b,name='output')

In [126]:
y_pred = pred(xtr, W, b)

3.Loss (Cost) Function [Mean square error]

In [127]:
def loss_calc(y, y_):
  return tf.reduce_mean(tf.square(y-y_),name='Loss')

In [128]:
loss = loss_calc(ytr, y_pred)
print(loss)

tf.Tensor(331.9501, shape=(), dtype=float32)


4.Function to train the Model

1.   Record all the mathematical steps to calculate Loss
2.   Calculate Gradients of Loss w.r.t weights and bias
3.   Update Weights and Bias based on gradients and learning rate to minimize loss

In [136]:
def fn_train(x,y,W,b,learn_rate):
  with tf.GradientTape() as t :
     t.watch([W,b])
     y_pred = pred(x,W,b)
     loss = loss_calc(y,y_pred)
  
  dc_dw, dc_db = t.gradient(loss, [W, b])
  W = W - learn_rate * dc_dw
  b = b - learn_rate * dc_db
  return W,b

## Train the model for 100 epochs 
1. Observe the training loss at every iteration
2. Observe Train loss at every 5th iteration

In [195]:
for i in range (100):
  W,b = fn_train(xtr, ytr, W,b, 0.01)
  y_pred = pred(xtr, W, b)
  train_loss = loss_calc(ytr, y_pred)
  if i % 5 == 0:
    print('Training loss at step: ', i, ' is ', train_loss)

Training loss at step:  0  is  tf.Tensor(285.47394, shape=(), dtype=float32)
Training loss at step:  5  is  tf.Tensor(285.47394, shape=(), dtype=float32)
Training loss at step:  10  is  tf.Tensor(285.47394, shape=(), dtype=float32)
Training loss at step:  15  is  tf.Tensor(285.47394, shape=(), dtype=float32)
Training loss at step:  20  is  tf.Tensor(285.47394, shape=(), dtype=float32)
Training loss at step:  25  is  tf.Tensor(285.47394, shape=(), dtype=float32)
Training loss at step:  30  is  tf.Tensor(285.47394, shape=(), dtype=float32)
Training loss at step:  35  is  tf.Tensor(285.47394, shape=(), dtype=float32)
Training loss at step:  40  is  tf.Tensor(285.47394, shape=(), dtype=float32)
Training loss at step:  45  is  tf.Tensor(285.47394, shape=(), dtype=float32)
Training loss at step:  50  is  tf.Tensor(285.47394, shape=(), dtype=float32)
Training loss at step:  55  is  tf.Tensor(285.47394, shape=(), dtype=float32)
Training loss at step:  60  is  tf.Tensor(285.47394, shape=(), dty

### Get the shapes and values of W and b

In [141]:
W

<tf.Tensor: id=10264, shape=(4, 1), dtype=float32, numpy=
array([[1.2170423],
       [1.5658139],
       [0.8839249],
       [2.3107197]], dtype=float32)>

In [143]:
W.shape

TensorShape([Dimension(4), Dimension(1)])

In [148]:
b

<tf.Tensor: id=10267, shape=(1,), dtype=float32, numpy=array([2.70448], dtype=float32)>

In [147]:
b.shape

TensorShape([Dimension(1)])

### Model Prediction on 1st Examples in Test Dataset

In [151]:
y_pred = pred(xte, W, b)
test_loss = loss_calc(yte, y_pred)

In [152]:
test_loss

<tf.Tensor: id=10291, shape=(), dtype=float32, numpy=54.32793>

## Classification using tf.Keras

In this exercise, we will build a Deep Neural Network using tf.Keras. We will use Iris Dataset for this exercise.

### Load the given Iris data using pandas (Iris.csv)

In [177]:
df = pd.read_csv('Iris.csv')

### Target set has different categories. So, Label encode them. And convert into one-hot vectors using get_dummies in pandas.

### Splitting the data into feature set and target set

In [178]:
X =  df.drop(["Id", "Species"], axis=1)
y =  df.pop("Species")

In [179]:
y = pd.get_dummies( y, columns = ['Species'] )

In [180]:
X.head()

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [181]:
y.head()

,Iris-setosa,Iris-versicolor,Iris-virginica
0,1,0,0
1,1,0,0
2,1,0,0
3,1,0,0
4,1,0,0


In [223]:
from sklearn.model_selection import train_test_split

In [233]:
train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.30)

###  Building Model in tf.keras

Build a Linear Classifier model  <br>
1.  Use Dense Layer  with input shape of 4 (according to the feature set) and number of outputs set to 3<br> 
2. Apply Softmax on Dense Layer outputs <br>
3. Use SGD as Optimizer
4. Use categorical_crossentropy as loss function 

In [234]:
model = tf.keras.Sequential([
  tf.keras.layers.Dense(10, activation=tf.nn.relu, input_shape=(4,)),  # input shape required
  #tf.keras.layers.Dense(10, activation=tf.nn.relu),
  tf.keras.layers.Dense(3, activation=tf.nn.softmax)
])

In [203]:
features = np.asarray(X)
labels = np.asarray(y)

In [184]:
features.shape

(150, 4)

In [235]:
model.compile(tf.keras.optimizers.SGD(), loss='categorical_crossentropy', metrics=['accuracy'])

### Model Training 

In [236]:
model.fit(train_x, train_y, epochs=100)

Epoch 1/100
105/105 [==============================] - 0s 554us/sample - loss: 1.8698 - acc: 0.1524
Epoch 2/100
105/105 [==============================] - 0s 41us/sample - loss: 1.2912 - acc: 0.4571
Epoch 3/100
105/105 [==============================] - 0s 37us/sample - loss: 1.2002 - acc: 0.5810
Epoch 4/100
105/105 [==============================] - 0s 31us/sample - loss: 1.1607 - acc: 0.4095
Epoch 5/100
105/105 [==============================] - 0s 33us/sample - loss: 1.0999 - acc: 0.6095
Epoch 6/100
105/105 [==============================] - 0s 35us/sample - loss: 1.0731 - acc: 0.4476
Epoch 7/100
105/105 [==============================] - 0s 46us/sample - loss: 1.0417 - acc: 0.6190
Epoch 8/100
105/105 [==============================] - 0s 34us/sample - loss: 0.9951 - acc: 0.4571
Epoch 9/100
105/105 [==============================] - 0s 29us/sample - loss: 0.9660 - acc: 0.5143
Epoch 10/100
105/105 [==============================] - 0s 31us/sample - loss: 0.9945 - acc: 0.5048
Epoch 11

105/105 [==============================] - 0s 41us/sample - loss: 0.3970 - acc: 0.9429
Epoch 84/100
105/105 [==============================] - 0s 34us/sample - loss: 0.3934 - acc: 0.9524
Epoch 85/100
105/105 [==============================] - 0s 35us/sample - loss: 0.3912 - acc: 0.9429
Epoch 86/100
105/105 [==============================] - 0s 34us/sample - loss: 0.3949 - acc: 0.8857
Epoch 87/100
105/105 [==============================] - 0s 36us/sample - loss: 0.3873 - acc: 0.9429
Epoch 88/100
105/105 [==============================] - 0s 30us/sample - loss: 0.3815 - acc: 0.9524
Epoch 89/100
105/105 [==============================] - 0s 32us/sample - loss: 0.3817 - acc: 0.9524
Epoch 90/100
105/105 [==============================] - 0s 33us/sample - loss: 0.3799 - acc: 0.9333
Epoch 91/100
105/105 [==============================] - 0s 38us/sample - loss: 0.3756 - acc: 0.9619
Epoch 92/100
105/105 [==============================] - 0s 31us/sample - loss: 0.3810 - acc: 0.9524
Epoch 93/100


### Model Prediction

In [237]:
model.evaluate(test_x, test_y)

45/45 [==============================] - 0s 499us/sample - loss: 0.2886 - acc: 0.9556


[0.2886086285114288, 0.95555556]

### Save the Model

In [231]:
model.save("iris_model.h5")

In [232]:
!ls

11_Iris.csv
Boston_Housing_Prices_tensorflow.py
Iris.csv
iris_model.h5
prices.csv
R6_Internal_Lab_UpdatedTF2_Prices_Iris.ipynb


### Build and Train a Deep Neural network with 2 hidden layer  - Optional - For Practice

Does it perform better than Linear Classifier? What could be the reason for difference in performance?

In [238]:
model = tf.keras.Sequential([
  tf.keras.layers.Dense(10, activation=tf.nn.relu, input_shape=(4,)),  # input shape required
  tf.keras.layers.Dense(10, activation=tf.nn.relu),
  tf.keras.layers.Dense(3, activation=tf.nn.softmax)
])

In [239]:
model.compile(tf.keras.optimizers.SGD(), loss='categorical_crossentropy', metrics=['accuracy'])

In [240]:
model.fit(train_x, train_y, epochs=100)

Epoch 1/100
105/105 [==============================] - 0s 597us/sample - loss: 2.0296 - acc: 0.3524
Epoch 2/100
105/105 [==============================] - 0s 30us/sample - loss: 1.4515 - acc: 0.3524
Epoch 3/100
105/105 [==============================] - 0s 31us/sample - loss: 1.2820 - acc: 0.3524
Epoch 4/100
105/105 [==============================] - 0s 32us/sample - loss: 1.2035 - acc: 0.3524
Epoch 5/100
105/105 [==============================] - 0s 38us/sample - loss: 1.1514 - acc: 0.3524
Epoch 6/100
105/105 [==============================] - 0s 32us/sample - loss: 1.1278 - acc: 0.3524
Epoch 7/100
105/105 [==============================] - 0s 40us/sample - loss: 1.1018 - acc: 0.3524
Epoch 8/100
105/105 [==============================] - 0s 36us/sample - loss: 1.0851 - acc: 0.3524
Epoch 9/100
105/105 [==============================] - 0s 33us/sample - loss: 1.0755 - acc: 0.3524
Epoch 10/100
105/105 [==============================] - 0s 31us/sample - loss: 1.0655 - acc: 0.3524
Epoch 11

105/105 [==============================] - 0s 36us/sample - loss: 0.6138 - acc: 0.6286
Epoch 84/100
105/105 [==============================] - 0s 30us/sample - loss: 0.6116 - acc: 0.6286
Epoch 85/100
105/105 [==============================] - 0s 31us/sample - loss: 0.6069 - acc: 0.6286
Epoch 86/100
105/105 [==============================] - 0s 37us/sample - loss: 0.6036 - acc: 0.6286
Epoch 87/100
105/105 [==============================] - 0s 33us/sample - loss: 0.6008 - acc: 0.6286
Epoch 88/100
105/105 [==============================] - 0s 36us/sample - loss: 0.5977 - acc: 0.6286
Epoch 89/100
105/105 [==============================] - 0s 35us/sample - loss: 0.5934 - acc: 0.6286
Epoch 90/100
105/105 [==============================] - 0s 35us/sample - loss: 0.5908 - acc: 0.6286
Epoch 91/100
105/105 [==============================] - 0s 40us/sample - loss: 0.5871 - acc: 0.6286
Epoch 92/100
105/105 [==============================] - 0s 28us/sample - loss: 0.5839 - acc: 0.6286
Epoch 93/100


In [241]:
model.evaluate(test_x, test_y)

45/45 [==============================] - 0s 587us/sample - loss: 0.4628 - acc: 0.7556


[0.4628284421232012, 0.75555557]